##### Copyright 2026 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 推理


Gemma 是一系列輕量級、最先進的開放模型，建立在與建立 [Gemini](https://deepmind.google/technologies/gemini/#introduction) 模型相同的研究與技術之上。Gemma 4 旨在成為全球最具效率的開放權重模型系列。

本文件示範如何使用 Gemma 4 的思考功能，在給出最終答案前產生推理過程。您將學會如何使用 Hugging Face [`transformers`](https://github.com/huggingface/transformers) 函式庫，為純文字與多模態（圖像＋文字）任務啟用思考模式，並解析輸出以將思考過程與答案分開。

## 安裝 Python 套件

安裝執行 Gemma 模型與發出請求所需的 Hugging Face 函式庫。

In [ ]:
%%bash
# Install PyTorch & other libraries
pip install torch accelerate

# Install the transformers library
pip install "transformers>=5.10.1"


## 載入模型

使用 [`transformers`](https://github.com/huggingface/transformers) 函式庫，如以下程式碼範例所示，使用 `AutoProcessor` 與 `AutoModelForImageTextToText` 類別建立 `processor` 與 `model` 的執行個體：

In [ ]:
MODEL_ID = "google/gemma-4-E2B-it"

from transformers import AutoProcessor, AutoModelForMultimodalLM

model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)


## 單次文字推論搭配思考模式

若要使用模型的思考功能產生回應，請傳入 `enable_thinking=True`；處理器會在提示中插入正確的思考詞元，指示模型在回應前先思考。
> **Note:** 我們已在 `gemma-4-12B-it`、`gemma-4-26B-A4B-it` 與 `gemma-4-31B-it` 的聊天範本中加入了空白思考詞元，可抑制思考停用時仍可能出現的「幽靈」思考頻道（thought channel），使模型輸出更穩定。

| 模型大小 | 思考狀態 | 範本結構／輸出 |
|---|---|---|
| **E2B/E4B** | **OFF** | `<\|turn>user\n[Prompt]<turn\|>\n<\|turn>model` |
| **E2B/E4B** | **ON** | `<\|turn>system\n<\|think\|><turn\|>\n<\|turn>user\n[Prompt]<turn\|>\n<\|turn>model` |
| **12B/26B/31B** | **OFF** | ⚠️ `<\|turn>user\n[Prompt]<turn\|>\n<\|turn>model\n<\|channel>thought\n<channel\|>` |
| **12B/26B/31B** | **ON** | `<\|turn>system\n<\|think\|><turn\|>\n<\|turn>user\n[Prompt]<turn\|>\n<\|turn>model` |

In [ ]:
from transformers import TextStreamer

message = [
    {
        "role": "user", "content": "What is the water formula?"
    }
]

text = processor.apply_chat_template(message, tokenize=False, add_generation_prompt=True, enable_thinking=True)
inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

streamer = TextStreamer(processor)
outputs = model.generate(**inputs, streamer=streamer, max_new_tokens=1024)


文字產生後，回應將包含以特殊詞元為界的推理區塊與最終答案。您可以使用 `parse_response` 工具函式，輕鬆將它們擷取為包含 `thinking` 與 `answer` 的字典。
> **Note:** 若要讓推理解析器正確運作，必須使用 `skip_special_tokens=False`。

In [ ]:
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
result = processor.parse_response(response)

for key, value in result.items():
  if key == "role":
    print(f"Role: {value}")
  elif key == "thinking":
    print(f"\n=== Thoughts ===\n{value}")
  elif key == "content":
    print(f"\n=== Answer ===\n{value}")
  elif key == "tool_calls":
    print(f"\n=== Tool Calls ===\n{value}")
  else:
    print(f"\n{key}: {value}...\n")


## 多輪對話範例與思考過程剝除

在多輪對話中妥善管理模型產生的思考內容，對於維持效能至關重要。

- **標準多輪對話（Standard Multi-Turn Conversations）**：在將對話歷史記錄傳回模型進行下一輪對話前，您必須移除（剝除）模型在上一輪產生的思考內容。若想在對話中途停用思考模式，可在剝除先前思考內容時一併移除 `<|think|>` 詞元。
- **函式呼叫（例外，Function Calling）**：如果單一模型回合涉及函式或工具呼叫，在函式呼叫之間**不得**移除思考內容。
- **維持對話歷史記錄（Maintaining Conversation History）**：歷史模型輸出只能包含最終回應。在下一輪使用者提問開始前，請確保先前回合產生的思考內容都沒有留在上下文視窗中。

In [ ]:
from transformers import TextStreamer

# Append the clean response to the message history
message.append({
    "role": "assistant",
    "content": result["content"]
})

# ==========================================
# TURN 2
# ==========================================
print("\n--- Turn 2 ---")
# Add the next user query to the history
message.append({
    "role": "user",
    "content": "What is its boiling point in Celsius?"
})

text = processor.apply_chat_template(message, tokenize=False, add_generation_prompt=True, enable_thinking=True)
inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

streamer = TextStreamer(processor)
outputs = model.generate(**inputs, streamer=streamer, max_new_tokens=1024)


In [ ]:
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
result = processor.parse_response(response)

for key, value in result.items():
  if key == "role":
    print(f"Role: {value}")
  elif key == "thinking":
    print(f"\n=== Thoughts ===\n{value}")
  elif key == "content":
    print(f"\n=== Answer ===\n{value}")
  elif key == "tool_calls":
    print(f"\n=== Tool Calls ===\n{value}")
  else:
    print(f"\n{key}: {value}...\n")


## 單次圖像推論

將思考模型用於視覺資料的程序非常類似。您可以將圖像作為 `messages` 陣列的一部分提供。只需確保將圖像與格式化的文字一同傳給處理器，模型便會在回應前對視覺輸入進行推理。

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

prompt = "What is shown in this image?"
image_url = "https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/apps/sample-data/GoldenGate.png"

# download image
!wget -q {image_url} -O image.png
image = Image.open("image.png")

# Display all images
print("=== Downloaded image ===")
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(image)
ax.set_title("Image 1")
ax.axis("off")
plt.tight_layout()
plt.show()

message = [
    {
        "role": "user", "content": [
          {"type": "image"},
          {"type": "text", "text": prompt}
        ]
    }
]

text = processor.apply_chat_template(message, tokenize=False, add_generation_prompt=True, enable_thinking=True)
inputs = processor(text=text, images=image, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=1024)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

result = processor.parse_response(response)

for key, value in result.items():
  if key == "role":
    print(f"Role: {value}")
  elif key == "thinking":
    print(f"\n=== Thoughts ===\n{value}")
  elif key == "content":
    print(f"\n=== Answer ===\n{value}")
  elif key == "tool_calls":
    print(f"\n=== Tool Calls ===\n{value}")
  else:
    print(f"\n{key}: {value}...\n")


## 摘要

在本指南中，您學會了如何使用 Gemma 4 模型的思考功能，在最終答案前產生推理過程。內容包括：

1. **啟用思考模式（Enabling thinking mode）**：在 `apply_chat_template` 中使用 `enable_thinking=True`。
2. **文字串流（Text streamer）**：使用 `TextStreamer` 即時觀察思考過程。
3. **解析合併輸出（Parsing the combined output）**：使用 `parse_response` 分離 `thinking` 與 `answer` 區塊。
4. **多模態任務（Multimodal tasks）**：將思考功能應用於多模態任務（圖像＋文字）。